# TP1 — Semana 3 · Experimentacao Comparativa
## RSNA 2023 - Screening Mammography Breast Cancer Detection

**Grupo:** G6 - Marcos Felipe Arruda Lima, Adrian Lucas Pinheiro Silva, Igor Holanda Costa

Este notebook continua a partir do artefato gerado nas Semanas 1-2
(`train_with_folds.csv`, com a coluna `fold` ja congelada) e cobre:

1. Extracao de tres familias de descritores: **textura (GLCM/Haralick)**,
   **gradiente (HOG)** e **radiomica (PyRadiomics)**.
2. Grade de experimentos descritor x modelo, com validacao cruzada
   (usando os folds ja congelados) e ajuste de hiperparametros.
3. Estudo de ablacao (contribuicao de cada descritor e de etapas do
   pre-processamento).
4. Consolidacao de tabelas e figuras para a secao de Resultados do artigo.

> **Importante:** a partir deste ponto, `fold` **nao pode ser recalculado**
> - ele ja esta congelado desde a Semana 2. Qualquer subamostragem feita
> aqui preserva o `fold` original de cada linha.


> **Nota de implementacao:** a familia "radiomica" abaixo NAO usa o pacote
> `pyradiomics` — ele nao compila de forma confiavel em ambientes Python
> recentes (Kaggle roda Python 3.11/3.12; o `pyradiomics` so tem suporte
> oficial ate ~3.9), o que comprometeria a reprodutibilidade exigida pelo
> TP1. Em vez disso, implementamos manualmente o mesmo tipo de descritor
> (estatisticas de primeira ordem da intensidade + propriedades de forma da
> ROI, o nucleo do que o PyRadiomics calcula) usando apenas `scipy` e
> `scikit-image`, ja disponiveis por padrao.

In [ ]:
!pip install pylibjpeg pylibjpeg-libjpeg

In [ ]:
# ==== Imports ====
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import pydicom
from pydicom.pixel_data_handlers.util import apply_voi_lut
import cv2

from scipy import stats
from skimage.feature import graycomatrix, graycoprops, hog
from skimage import measure

from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, balanced_accuracy_score,
)

sns.set_theme(style="whitegrid")


In [ ]:
SEED = 42
np.random.seed(SEED)

DATA_DIR = "/kaggle/input/competitions/rsna-breast-cancer-detection"
OUTPUT_DIR = "/kaggle/working"
IMAGES_DIR = os.path.join(DATA_DIR, "train_images")

FOLDS_CSV_PATH = "/kaggle/input/notebooks/igorholanda/gp6-screening-mammography-breast-cancer/train_with_folds.csv"  # ajustar com o resultado do find

TARGET_COL = "cancer"
GROUP_COL = "patient_id"

NEG_PER_POS = 2

fold_df = pd.read_csv(FOLDS_CSV_PATH)

positives = fold_df[fold_df[TARGET_COL] == 1]
negatives = fold_df[fold_df[TARGET_COL] == 0].sample(
    n=min(len(positives) * NEG_PER_POS, (fold_df[TARGET_COL] == 0).sum()),
    random_state=SEED,
)
sample_df = pd.concat([positives, negatives]).sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"Amostra da Semana 3: {len(sample_df)} imagens "
      f"({positives[TARGET_COL].sum()} positivos, {len(negatives)} negativos)")
print(sample_df.groupby("fold")[TARGET_COL].agg(["count", "mean"]))

## 1. Funcoes de pre-processamento (reaproveitadas da Semana 1-2)

Repetidas aqui para que este notebook seja executavel de forma independente
(criterio de reprodutibilidade).

In [ ]:
def load_dicom_image(path, apply_voi=True):
    dcm = pydicom.dcmread(path)
    img = dcm.pixel_array.astype(np.float32)
    if apply_voi:
        img = apply_voi_lut(img, dcm)
    if getattr(dcm, "PhotometricInterpretation", "") == "MONOCHROME1":
        img = img.max() - img
    img = img - img.min()
    if img.max() > 0:
        img = img / img.max()
    return (img * 255).astype(np.uint8)


def remove_background(img):
    _, mask = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel = np.ones((15, 15), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if n_labels <= 1:
        return img, mask
    largest_label = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    breast_mask = (labels == largest_label).astype(np.uint8) * 255
    result = cv2.bitwise_and(img, img, mask=breast_mask)
    return result, breast_mask


def crop_to_roi(img, mask, margin=10):
    ys, xs = np.where(mask > 0)
    if len(xs) == 0:
        return img, mask
    x0, x1 = max(xs.min() - margin, 0), min(xs.max() + margin, img.shape[1])
    y0, y1 = max(ys.min() - margin, 0), min(ys.max() + margin, img.shape[0])
    return img[y0:y1, x0:x1], mask[y0:y1, x0:x1]


def enhance_contrast(img, clip_limit=1.0, tile_grid_size=(8, 8)):
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    return clahe.apply(img)


def preprocess_image(path, target_size=(256, 256), apply_clahe=True, apply_bg_removal=True):
    """Pipeline de pre-processamento. Os parametros apply_clahe/apply_bg_removal
    existem especificamente para o estudo de ablacao da secao 5."""
    img = load_dicom_image(path)
    if apply_bg_removal:
        img, mask = remove_background(img)
        img, mask = crop_to_roi(img, mask)
    else:
        mask = np.ones_like(img) * 255
    if apply_clahe:
        img = enhance_contrast(img)
    img = cv2.resize(img, target_size, interpolation=cv2.INTER_AREA)
    mask = cv2.resize(mask, target_size, interpolation=cv2.INTER_NEAREST)
    return img, mask


## 2. Extracao de descritores

Tres familias, conforme exigido no enunciado (minimo tres):

- **Textura - GLCM/Haralick:** contraste, correlacao, energia, homogeneidade
  e ASM, calculados em 4 angulos e media entre eles.
- **Gradiente - HOG:** captura a espiculacao das massas malignas.
- **Radiomica - primeira ordem + forma (inspirado em IBSI/PyRadiomics):**
  estatisticas de intensidade da ROI (media, variancia, assimetria,
  curtose, energia, entropia, percentis) e propriedades geometricas da
  regiao segmentada (area, perimetro, excentricidade, solidez).

In [ ]:
def extract_glcm_features(img, distances=(1, 3), angles=(0, np.pi/4, np.pi/2, 3*np.pi/4)):
    glcm = graycomatrix(img, distances=list(distances), angles=list(angles),
                         levels=256, symmetric=True, normed=True)
    props = ["contrast", "correlation", "energy", "homogeneity", "ASM", "dissimilarity"]
    feats = {}
    for prop in props:
        values = graycoprops(glcm, prop)
        feats[f"glcm_{prop}_mean"] = float(np.mean(values))
        feats[f"glcm_{prop}_std"] = float(np.std(values))
    return feats


def extract_hog_features(img, pixels_per_cell=(32, 32), cells_per_block=(2, 2)):
    features = hog(
        img, orientations=9, pixels_per_cell=pixels_per_cell,
        cells_per_block=cells_per_block, block_norm="L2-Hys",
        visualize=False, feature_vector=True,
    )
    # Reduzimos para estatisticas resumo (media/desvio/percentis) em vez do
    # vetor HOG bruto, para manter a dimensionalidade controlada entre
    # imagens de tamanhos variados e evitar overfitting em alta dimensao.
    return {
        "hog_mean": float(np.mean(features)),
        "hog_std": float(np.std(features)),
        "hog_p90": float(np.percentile(features, 90)),
        "hog_max": float(np.max(features)),
    }


def extract_radiomics_features(img, mask):
    """Descritores radiomicos "leves" (primeira ordem + forma), inspirados
    no conjunto IBSI/PyRadiomics, implementados sem a dependencia pyradiomics
    (instavel em Python 3.11+). Cobrem a mesma ideia central: estatisticas
    de intensidade da ROI e propriedades geometricas da regiao segmentada.
    """
    roi_pixels = img[mask > 0]
    if roi_pixels.size < 10:
        return {}

    hist, _ = np.histogram(roi_pixels, bins=256, range=(0, 255))
    hist_norm = hist / hist.sum()

    feats = {
        "rad_mean": float(np.mean(roi_pixels)),
        "rad_std": float(np.std(roi_pixels)),
        "rad_variance": float(np.var(roi_pixels)),
        "rad_skewness": float(stats.skew(roi_pixels)),
        "rad_kurtosis": float(stats.kurtosis(roi_pixels)),
        "rad_energy": float(np.sum(roi_pixels.astype(np.float64) ** 2)),
        "rad_entropy": float(stats.entropy(hist_norm + 1e-9)),
        "rad_uniformity": float(np.sum(hist_norm ** 2)),
        "rad_p10": float(np.percentile(roi_pixels, 10)),
        "rad_p90": float(np.percentile(roi_pixels, 90)),
    }

    props_list = measure.regionprops((mask > 0).astype(np.uint8))
    if props_list:
        props = props_list[0]
        feats.update({
            "rad_area": float(props.area),
            "rad_perimeter": float(props.perimeter),
            "rad_eccentricity": float(props.eccentricity),
            "rad_solidity": float(props.solidity),
            "rad_extent": float(props.extent),
        })

    return feats


def extract_all_features(path, apply_clahe=True, apply_bg_removal=True):
    img, mask = preprocess_image(path, apply_clahe=apply_clahe, apply_bg_removal=apply_bg_removal)
    feats = {}
    feats.update(extract_glcm_features(img))
    feats.update(extract_hog_features(img))
    feats.update(extract_radiomics_features(img, mask))
    return feats


## 3. Extracao em lote (com cache em disco)

A extracao completa e custosa - rodamos uma vez sobre a amostra definida
na Secao 0 e salvamos o resultado, para nao precisar refazer a cada
execucao do notebook.

In [ ]:
FEATURES_PATH = os.path.join(OUTPUT_DIR, "features_week3.csv")

if os.path.exists(FEATURES_PATH):
    features_df = pd.read_csv(FEATURES_PATH)
    print(f"Features carregadas do cache: {features_df.shape}")
else:
    rows = []
    errors = 0
    for _, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc="Extraindo descritores"):
        path = os.path.join(IMAGES_DIR, str(row[GROUP_COL]), f"{row['image_id']}.dcm")
        try:
            feats = extract_all_features(path)
            feats[GROUP_COL] = row[GROUP_COL]
            feats["image_id"] = row["image_id"]
            feats[TARGET_COL] = row[TARGET_COL]
            feats["fold"] = row["fold"]
            rows.append(feats)
        except Exception:
            errors += 1
            continue

    features_df = pd.DataFrame(rows)
    features_df.to_csv(FEATURES_PATH, index=False)
    print(f"Extracao concluida: {len(features_df)} imagens processadas, {errors} erros.")

feature_groups = {
    "glcm": [c for c in features_df.columns if c.startswith("glcm_")],
    "hog": [c for c in features_df.columns if c.startswith("hog_")],
    "radiomics": [c for c in features_df.columns if c.startswith("rad_")],
}
feature_groups["all"] = feature_groups["glcm"] + feature_groups["hog"] + feature_groups["radiomics"]
print({k: len(v) for k, v in feature_groups.items()})


## 4. Grade de experimentos: descritor x modelo

Validacao usando os folds ja congelados - o ajuste de hiperparametros ocorre
apenas dentro do treino de cada fold, nunca olhando a validacao. O
desbalanceamento e tratado via `class_weight='balanced'` em todos os
modelos que suportam o parametro.

In [ ]:
MODEL_GRID = {
    "SVM_linear": (
        SVC(kernel="linear", probability=True, class_weight="balanced", random_state=SEED),
        {"clf__C": [0.1, 1, 10]},
    ),
    "SVM_rbf": (
        SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=SEED),
        {"clf__C": [0.1, 1, 10], "clf__gamma": ["scale", "auto"]},
    ),
    "RandomForest": (
        RandomForestClassifier(class_weight="balanced", random_state=SEED),
        {"clf__n_estimators": [200, 400], "clf__max_depth": [None, 10, 20]},
    ),
    "XGBoost": (
        XGBClassifier(eval_metric="logloss", random_state=SEED),
        {"clf__n_estimators": [200, 400], "clf__max_depth": [3, 6], "clf__learning_rate": [0.05, 0.1]},
    ),
    "kNN": (
        KNeighborsClassifier(),
        {"clf__n_neighbors": [5, 11, 21]},
    ),
    "LogisticRegression": (
        LogisticRegression(class_weight="balanced", max_iter=2000, random_state=SEED),
        {"clf__C": [0.1, 1, 10]},
    ),
}


def run_experiment(X, y, fold_labels, model, param_grid):
    """Roda um modelo com ajuste de hiperparametros por fold, usando o
    fold como conjunto de validacao em cada iteracao."""
    aucs, aps, bals = [], [], []
    for fold in sorted(fold_labels.unique()):
        train_idx = (fold_labels != fold).values
        val_idx = (fold_labels == fold).values

        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        pipe = Pipeline([("scaler", StandardScaler()), ("clf", model)])
        gs = GridSearchCV(pipe, param_grid, scoring="average_precision", cv=3, n_jobs=-1)
        gs.fit(X_train, y_train)

        y_score = gs.predict_proba(X_val)[:, 1]
        y_pred = gs.predict(X_val)

        aucs.append(roc_auc_score(y_val, y_score))
        aps.append(average_precision_score(y_val, y_score))
        bals.append(balanced_accuracy_score(y_val, y_pred))

    return {
        "auc_roc_mean": np.mean(aucs), "auc_roc_std": np.std(aucs),
        "auc_pr_mean": np.mean(aps), "auc_pr_std": np.std(aps),
        "bal_acc_mean": np.mean(bals), "bal_acc_std": np.std(bals),
    }


In [ ]:
results = []

for descriptor_name, cols in feature_groups.items():
    if len(cols) == 0:
        continue
    X = features_df[cols].fillna(0).values
    y = features_df[TARGET_COL].values
    fold_labels = features_df["fold"]

    for model_name, (model, param_grid) in MODEL_GRID.items():
        print(f"Rodando {descriptor_name} x {model_name}...")
        metrics = run_experiment(X, y, fold_labels, model, param_grid)
        metrics["descriptor"] = descriptor_name
        metrics["model"] = model_name
        results.append(metrics)

results_df = pd.DataFrame(results)
results_df.to_csv(os.path.join(OUTPUT_DIR, "results_week3.csv"), index=False)
results_df.sort_values("auc_pr_mean", ascending=False)


In [ ]:
# ==== Matriz de confusao real (rapido - sem carregar imagens) ====
from sklearn.metrics import confusion_matrix

best_descriptor, best_model_name = "radiomics", "LogisticRegression"
cols = feature_groups[best_descriptor]
X = features_df[cols].fillna(0).values
y = features_df[TARGET_COL].values
fold_labels = features_df["fold"]

model, param_grid = MODEL_GRID[best_model_name]
all_scores = np.zeros(len(y))

for fold in sorted(fold_labels.unique()):
    train_idx = (fold_labels != fold).values
    val_idx = (fold_labels == fold).values
    pipe = Pipeline([("scaler", StandardScaler()), ("clf", model)])
    gs = GridSearchCV(pipe, param_grid, scoring="average_precision", cv=3, n_jobs=-1)
    gs.fit(X[train_idx], y[train_idx])
    all_scores[val_idx] = gs.predict_proba(X[val_idx])[:, 1]

y_pred = (all_scores >= 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()

print(f"VP (verdadeiros positivos): {tp}")
print(f"VN (verdadeiros negativos): {tn}")
print(f"FP (falsos positivos): {fp}")
print(f"FN (falsos negativos): {fn}")
print(f"Sensibilidade (recall): {tp/(tp+fn):.3f}")
print(f"Especificidade: {tn/(tn+fp):.3f}")

## 5. Estudo de ablacao

Duas perguntas separadas:

1. **Quais familias de descritores mais contribuem?** - ja respondida
   pela propria grade acima (comparacao `glcm` vs `hog` vs `radiomics` vs
   `all`, para um mesmo modelo).
2. **Quais etapas do pipeline de pre-processamento mais contribuem?** -
   testamos abaixo, fixando o melhor descritor + modelo da grade e
   comparando com/sem remocao de fundo e com/sem CLAHE.

In [ ]:
best_row = results_df.sort_values("auc_pr_mean", ascending=False).iloc[0]
print("Melhor combinacao da grade:", best_row["descriptor"], "+", best_row["model"])

ablation_configs = {
    "pipeline_completo": {"apply_bg_removal": True, "apply_clahe": True},
    "sem_remocao_fundo": {"apply_bg_removal": False, "apply_clahe": True},
    "sem_clahe": {"apply_bg_removal": True, "apply_clahe": False},
    "sem_nenhum": {"apply_bg_removal": False, "apply_clahe": False},
}

ablation_rows = []
for config_name, kwargs in ablation_configs.items():
    rows = []
    for _, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc=config_name):
        path = os.path.join(IMAGES_DIR, str(row[GROUP_COL]), f"{row['image_id']}.dcm")
        try:
            feats = extract_all_features(path, **kwargs)
            feats[TARGET_COL] = row[TARGET_COL]
            feats["fold"] = row["fold"]
            rows.append(feats)
        except Exception:
            continue

    df_ablation = pd.DataFrame(rows)
    cols = [c for c in df_ablation.columns if c not in (TARGET_COL, "fold")]
    X = df_ablation[cols].fillna(0).values
    y = df_ablation[TARGET_COL].values
    fold_labels = df_ablation["fold"]

    model, param_grid = MODEL_GRID[best_row["model"]]
    metrics = run_experiment(X, y, fold_labels, model, param_grid)
    metrics["config"] = config_name
    ablation_rows.append(metrics)

ablation_df = pd.DataFrame(ablation_rows)
ablation_df.to_csv(os.path.join(OUTPUT_DIR, "ablation_week3.csv"), index=False)
ablation_df[["config", "auc_pr_mean", "auc_pr_std", "auc_roc_mean"]]


## 6. Tabelas e figuras consolidadas

In [ ]:
# Tabela descritor x modelo (AUC-PR, metrica principal dado o desbalanceamento)
pivot_table = results_df.pivot(index="descriptor", columns="model", values="auc_pr_mean")
print(pivot_table.round(4))

fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(pivot_table, annot=True, fmt=".3f", cmap="viridis", ax=ax)
ax.set_title("AUC-PR medio por descritor x modelo (Semana 3)")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "fig_heatmap_descritor_modelo.png"), dpi=150)
plt.show()


In [ ]:
# Comparacao com o baseline trivial (Semana 2) - AUC-PR ~ prevalencia (~0.02)
TRIVIAL_AUC_PR = features_df[TARGET_COL].mean()

fig, ax = plt.subplots(figsize=(8, 5))
results_df.groupby("descriptor")["auc_pr_mean"].max().plot(kind="bar", ax=ax, color="#4C72B0")
ax.axhline(TRIVIAL_AUC_PR, color="red", linestyle="--", label=f"Baseline trivial (AUC-PR={TRIVIAL_AUC_PR:.4f})")
ax.set_title("Melhor AUC-PR por familia de descritor vs. baseline trivial")
ax.set_ylabel("AUC-PR")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "fig_descritor_vs_baseline.png"), dpi=150)
plt.show()


In [ ]:
# Grafico de ablacao do pipeline
fig, ax = plt.subplots(figsize=(8, 5))
ablation_df.set_index("config")["auc_pr_mean"].plot(kind="bar", ax=ax, yerr=ablation_df["auc_pr_std"], color="#55A868")
ax.set_title(f"Ablacao do pipeline - {best_row['descriptor']} + {best_row['model']}")
ax.set_ylabel("AUC-PR")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "fig_ablacao_pipeline.png"), dpi=150)
plt.show()


## Agradecimentos e nota de IA (Semana 4)

-Agradecemos por ter chegado até ao fim deste código!

---
**Nota sobre uso de IA generativa:** Utilizamos a IA (Claude/Gemini) como parceira de análise e suporte na hora de criar, ajustar e refatorar o código, garantindo que o resultado final fosse limpo e 100% reprodutível.